# JSON - Python

All 9 Python examples from [docs/json.md](https://platob.github.io/yggdryl/json/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import json

value = {"symbol": "AAPL", "quantity": 100}

encoded = json.dumps(value)
assert encoded == b'{"symbol":"AAPL","quantity":100}'
assert json.loads(encoded) == value
assert json.loads('{"symbol":"AAPL","quantity":100}') == value
assert json.loads(encoded)["symbol"] == "AAPL"

## Values JSON has no syntax for

In [ ]:
import math

from yggdryl import json

value = {"payload": b"\x00\x01\xff", "big": 2**127, "ratio": math.nan}

encoded = json.dumps(value)
assert b'"$yggdryl"' in encoded

decoded = json.loads(encoded)
assert decoded["payload"] == b"\x00\x01\xff"
assert decoded["big"] == 2**127
assert math.isnan(decoded["ratio"])

## Readers and writers

In [ ]:
import io

from yggdryl import json

value = {"symbol": "AAPL"}

target = io.BytesIO()
json.dump(value, target)
assert json.load(io.BytesIO(target.getvalue())) == value

reason = None
try:
    json.loads('{"symbol":"AAPL"} 42')
except ValueError as error:
    reason = str(error)
assert reason == "invalid json data at byte 18: trailing characters after JSON value"

## Newline-delimited JSON

In [ ]:
from yggdryl import json

rows = [{"id": 1}, {"id": 2}]

encoded = json.dumps_all(rows)
assert encoded == b'{"id":1}\n{"id":2}\n'
assert list(json.loads_all(encoded)) == rows

# Blank and CRLF-terminated lines are skipped; two values on one are not.
assert list(json.loads_all(b'{"id":1}\r\n\n{"id":2}\n')) == rows

reason = None
try:
    list(json.loads_all(b'{"id":1} {"id":2}\n'))
except ValueError as error:
    reason = str(error)
assert reason == "invalid json data at byte 9: trailing characters after JSON value"

In [ ]:
import pathlib
import tempfile

from yggdryl import json

with tempfile.TemporaryDirectory() as directory:
    path = pathlib.Path(directory) / "rows.jsonl"
    json.dump_all([{"id": 1}, {"id": 2}, {"id": 3}], path)

    rows = json.load_all(path)
    assert iter(rows) is rows
    assert next(rows) == {"id": 1}
    assert list(rows) == [{"id": 2}, {"id": 3}]

## Laying out a dump

In [ ]:
from yggdryl import DataType, Field, json

field = Field("id", "int64", nullable=False)

# `indent=None` is compact - the default, as `json.dumps` has it.
assert "\n" not in field.to_json()
assert field.to_json(indent=2).startswith('{\n  "name": "id",')

# Round-trip and idempotence hold for every setting.
for indent in (None, 2, 4):
    text = field.to_json(indent=indent)
    assert Field.from_json(text) == field
    assert field.to_json(indent=indent) == text

## Limits

In [ ]:
from yggdryl import json

reason = None
try:
    json.loads(b"[" * 129 + b"0" + b"]" * 129)
except ValueError as error:
    reason = str(error)
assert reason == "invalid json data at byte 128: nesting depth limit exceeded"

reason = None
try:
    json.dumps_all({"id": index} for index in range(1025))
except ValueError as error:
    reason = str(error)
assert reason == "codec collection exceeds the 1024-document limit"

## Failures carry a byte offset

In [ ]:
from yggdryl import json

reason = None
try:
    json.loads('{"symbol":"AAPL","symbol":"MSFT"}')
except ValueError as error:
    reason = str(error)
assert reason == "invalid json data at byte 17: JSON object contains a duplicate key"

reason = None
try:
    list(json.loads_all(b'{"id":1}\n{bad}\n'))
except ValueError as error:
    reason = str(error)
assert reason == "invalid json data at byte 10: JSON object key must be a string"

## A compound filename carries the coding

In [ ]:
import pathlib
import tempfile

from yggdryl import json

with tempfile.TemporaryDirectory() as directory:
    path = pathlib.Path(directory) / "trades.json"

    json.dump({"symbol": "AAPL"}, path)
    assert json.load(path) == {"symbol": "AAPL"}

    # A str that is not an existing file is content, not a location.
    assert json.load('{"symbol":"AAPL"}') == {"symbol": "AAPL"}